In [ ]:
Analysis of aging-related differential gut microbial changes 

MaAsLin2 (version 1.16.0) was employed to perform aging-related differential species, function analysis. In the discovery cohort, we used MaAsLin2 (version 1.16.0) in R, which finds associations between enterotype-independent microbial features and age, and MaAsLin2 uses a transformed generalized linear model to associate each feature iteratively with covariates of interest.
Feature ≈ age + enterotype + sex + BMI
In the validation cohort, we used MaAsLin2 (version 1.16.0) in R, which finds associations between enterotype-independent microbial features and age, and MaAsLin2 uses a transformed generalized linear model to associate each feature iteratively with covariates of interest.
Feature ≈ age + enterotype + sex +BMI + Center
We used MaAsLin2 (version 1.16.0) in R, which finds associations between enterotype-stratified microbial features and age, and MaAsLin2 uses a transformed generalized linear model to associate each feature iteratively with covariates of interest.
Feature ≈ age + BMI + sex
We used MaAsLin2 (version 1.16.0) in R, which finds associations between sex-stratified microbial features and age, and MaAsLin2 uses a transformed generalized linear model to associate each feature iteratively with covariates of interest.
Feature ≈ age + BMI + enterotype
Here, we used a variance-stabilizing log transformation plus a small pseudocount of half the minimum feature value for microbial relative abundances (total sum scaling), then it models each microbial feature as a function of the patient’s age and adjust the resulting P values for multiple hypothesis tests, using a Benjamini-Hochberg correction. Consistent with prior studies, a false discovery rate (FDR, q value) less than 0.1 was set as the threshold for statistical significance. The details of aging-related differential species and functions are listed in Supplementary Table 4.

In [ ]:
rm(list=ls())
library(tidyverse)
library(ggplot2)
library(ggsci)
library(dplyr)
library(patchwork)
library(Maaslin2)

count_file_name="E:/QZ/QZ_metagenomics/01_Upstream/04_Result/01_metaphlan/merged_abundance_table_read_count_species.txt"
df_input_data = read.table(file             = count_file_name,
                           header           = TRUE,
                           sep              = "\t", 
                           row.names        = 1,
                           stringsAsFactors = FALSE,
                           check.names = F)
df_input_data[1:5, 1:5]

df_input_metadata = read.table(file             = "E:/QZ/QZ_metagenomics/04_Merge/Sample_metadata.csv", 
                               header           = TRUE, 
                               sep              = ",", 
                               row.names        = 1,
                               stringsAsFactors = FALSE)
rownames(df_input_metadata)

table(df_input_metadata$Group)
df_input_metadata<-df_input_metadata%>%filter(Group=="Control")

Enterotype_cluster = read.csv(
  "E:/QZ/QZ_metagenomics/04_Merge/02_Read_based/01_Taxonomy/Enterotype/2/data.cluster.csv")
names(Enterotype_cluster)[2]<-"Enterotype"
table(Enterotype_cluster$Enterotype)

df_input_metadata<-merge(Enterotype_cluster,df_input_metadata,
            by.x = "X", by.y = "Name_metaphlan")

rownames(df_input_metadata)<-df_input_metadata$X
table(df_input_metadata$Group)
table(df_input_metadata$Sex)
table(df_input_metadata$Batch)
intersect(rownames(df_input_metadata),colnames(df_input_data))
df_input_data<-df_input_data[,intersect(rownames(df_input_metadata),colnames(df_input_data))]
head(df_input_metadata)
colnames(df_input_metadata)
table(df_input_metadata$Sex)
fit_data = Maaslin2(input_data     = df_input_data,
                    input_metadata = df_input_metadata,
                    min_prevalence = 0.05,
                    # normalization  = "NONE",
                    # standardize = FALSE,
                    output         = "Result", 
                    # cores = 4 ,
                    fixed_effects  = c("Age",
                                       "Sex",
                                       "Enterotype",
                                       "BMI"))
